In [ ]:
"""
DATA CHALLENGE — HEIGHT
Hypsometric relationship (DBH x Height) and estimation of unmeasured
tree heights, Stand 8 (eucalyptus inventory)

Course: Forest Mensuration
Author: [Daniyal Hussain]

"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)

# -----------------------------------------------------------------------
# 0. PATHS — relative to the repository root
# -----------------------------------------------------------------------
DATA_DIR = "data"
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Folder '{DATA_DIR}' not found. Run this script from the repository "
        "root (the folder containing data/, notebooks/, outputs/, report/)."
    )

xlsx_candidates = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(".xlsx")]
if len(xlsx_candidates) == 0:
    raise FileNotFoundError(
        f"No .xlsx file found in '{DATA_DIR}'. Copy your inventory spreadsheet "
        "into that folder and re-run."
    )
elif len(xlsx_candidates) > 1:
    print(f"WARNING: multiple .xlsx files found, using the first one: {xlsx_candidates}")

INPUT_FILE = os.path.join(DATA_DIR, xlsx_candidates[0])
print(f"Loading input file: {INPUT_FILE}")

# -----------------------------------------------------------------------
# 0b. DATA LOADING AND PREPARATION
# -----------------------------------------------------------------------
df = pd.read_excel(INPUT_FILE)
df = df.rename(columns={
    'PARCELA': 'plot',
    'FILEIRA': 'row',
    'ÁRVORE': 'tree',
    'DAP': 'dbh',
    'HT': 'ht',
})

print(f"Dataset loaded: {df.shape[0]} trees, {df['plot'].nunique()} plots")

# -----------------------------------------------------------------------
# TASK 1 — Confirm the height sampling pattern
# -----------------------------------------------------------------------
df['ht_measured'] = df['ht'] > 0

print("\n" + "=" * 70)
print("[Task 1] Height sampling pattern — which rows were actually measured?")
print("=" * 70)
crosstab = pd.crosstab(df['row'], df['ht_measured'])
print(crosstab)

measured_rows = sorted(df.loc[df['ht_measured'], 'row'].unique())
unmeasured_rows = sorted(df.loc[~df['ht_measured'], 'row'].unique())
print(f"\nRows with height measured:   {measured_rows}")
print(f"Rows WITHOUT height measured: {unmeasured_rows}")
print(f"Total measured trees:   {df['ht_measured'].sum()}")
print(f"Total unmeasured trees: {(~df['ht_measured']).sum()}")

consistent = crosstab.apply(lambda r: (r == 0).sum() == 1, axis=1).all()
print(f"\nPattern is 100% consistent per row (no mixed rows): {consistent}")

# -----------------------------------------------------------------------
# TASK 2 — Fit hypsometric models (linear and log-log) using only
#          measured trees, and evaluate fit quality
# -----------------------------------------------------------------------
measured = df[df['ht_measured']].copy()
unmeasured = df[~df['ht_measured']].copy()

print("\n" + "=" * 70)
print(f"[Task 2] Fitting hypsometric models on n={len(measured)} measured trees")
print("=" * 70)


def fit_ols(x, y):
    """Simple OLS: y = b0 + b1*x. Returns (b0, b1, fitted, residuals, r2, rmse)."""
    n = len(x)
    b1 = np.cov(x, y, ddof=1)[0, 1] / np.var(x, ddof=1)
    b0 = y.mean() - b1 * x.mean()
    fitted = b0 + b1 * x
    resid = y - fitted
    ss_res = np.sum(resid ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    rmse = np.sqrt(ss_res / n)
    return b0, b1, fitted, resid, r2, rmse


# --- Model A: Linear (HT = b0 + b1*DBH) ---
b0_lin, b1_lin, fitted_lin, resid_lin, r2_lin, rmse_lin = fit_ols(
    measured['dbh'].values, measured['ht'].values
)
print(f"\nModel A — Linear:  HT = {b0_lin:.4f} + {b1_lin:.4f} * DBH")
print(f"  R^2 = {r2_lin:.4f} | RMSE = {rmse_lin:.4f} m")

# --- Model B: Log-log (ln(HT) = b0 + b1*ln(DBH)) ---
ln_dbh = np.log(measured['dbh'].values)
ln_ht = np.log(measured['ht'].values)
b0_log, b1_log, fitted_ln, resid_ln, r2_log, rmse_log_scale = fit_ols(ln_dbh, ln_ht)

# Back-transform to original scale (Baskerville correction factor — a naive
# exp() back-transform underestimates the mean on the original scale)
see_log = np.sqrt(np.sum(resid_ln ** 2) / (len(resid_ln) - 2))
correction_factor = np.exp((see_log ** 2) / 2)
fitted_log_backtransformed = correction_factor * np.exp(fitted_ln)
resid_log_original_scale = measured['ht'].values - fitted_log_backtransformed
ss_res_log = np.sum(resid_log_original_scale ** 2)
ss_tot_log = np.sum((measured['ht'].values - measured['ht'].values.mean()) ** 2)
r2_log_original = 1 - ss_res_log / ss_tot_log
rmse_log_original = np.sqrt(ss_res_log / len(measured))

print(f"\nModel B — Log-log: ln(HT) = {b0_log:.4f} + {b1_log:.4f} * ln(DBH)")
print(f"  Back-transformation correction factor (Baskerville): {correction_factor:.4f}")
print(f"  R^2 (log scale)      = {r2_log:.4f}")
print(f"  R^2 (original scale, after back-transform) = {r2_log_original:.4f}")
print(f"  RMSE (original scale) = {rmse_log_original:.4f} m")

print("\n--- Model comparison summary ---")
print(f"{'Model':<12}{'R2':>10}{'RMSE (m)':>12}")
print(f"{'Linear':<12}{r2_lin:>10.4f}{rmse_lin:>12.4f}")
print(f"{'Log-log':<12}{r2_log_original:>10.4f}{rmse_log_original:>12.4f}")

best_model = "Linear" if rmse_lin <= rmse_log_original else "Log-log"
print(f"\nBest model by RMSE: {best_model}")

# --- Scatter plot with both fitted curves ---
dbh_range = np.linspace(measured['dbh'].min(), df['dbh'].max(), 200)
ht_pred_lin_range = b0_lin + b1_lin * dbh_range
ht_pred_log_range = correction_factor * np.exp(b0_log + b1_log * np.log(dbh_range))

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(measured['dbh'], measured['ht'], color='#2E5A88', alpha=0.7,
           edgecolor='black', label='Measured trees (rows 3, 4, 5)', zorder=3)
ax.plot(dbh_range, ht_pred_lin_range, color='#C0392B', linewidth=2,
         label=f'Linear fit (R²={r2_lin:.3f})')
ax.plot(dbh_range, ht_pred_log_range, color='#27924A', linewidth=2, linestyle='--',
         label=f'Log-log fit (R²={r2_log_original:.3f})')
ax.set_xlabel('DBH (cm)')
ax.set_ylabel('Total height (m)')
ax.set_title('Hypsometric relationship — Stand 8 (Eucalyptus)')
ax.legend()
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/hypsometric_scatter_fit.png", dpi=150)
plt.close(fig)

# --- Residual diagnostic plots (both models) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(fitted_lin, resid_lin, color='#C0392B', alpha=0.7, edgecolor='black')
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_xlabel('Fitted HT (m)')
axes[0].set_ylabel('Residual (m)')
axes[0].set_title('Linear model — residuals vs fitted')

axes[1].scatter(fitted_log_backtransformed, resid_log_original_scale, color='#27924A',
                 alpha=0.7, edgecolor='black')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Fitted HT (m)')
axes[1].set_ylabel('Residual (m)')
axes[1].set_title('Log-log model — residuals vs fitted (original scale)')

fig.suptitle('Residual analysis — hypsometric models', fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(f"{OUT_DIR}/hypsometric_residuals.png", dpi=150)
plt.close(fig)

print(f"\nPlots saved:")
print(f"  - {OUT_DIR}/hypsometric_scatter_fit.png")
print(f"  - {OUT_DIR}/hypsometric_residuals.png")

# -----------------------------------------------------------------------
# TASK 3 — Apply the fitted model to unmeasured trees, produce a
#          complete dataset (215 trees, all with height)
# -----------------------------------------------------------------------
print("\n" + "=" * 70)
print(f"[Task 3] Applying '{best_model}' model to estimate height for "
      f"{len(unmeasured)} unmeasured trees")
print("=" * 70)

df_complete = df.copy()
df_complete['ht_source'] = np.where(df_complete['ht_measured'], 'measured', 'estimated')
df_complete['ht_final'] = df_complete['ht']

mask_unmeasured = ~df_complete['ht_measured']
if best_model == "Linear":
    df_complete.loc[mask_unmeasured, 'ht_final'] = (
        b0_lin + b1_lin * df_complete.loc[mask_unmeasured, 'dbh']
    )
else:
    df_complete.loc[mask_unmeasured, 'ht_final'] = correction_factor * np.exp(
        b0_log + b1_log * np.log(df_complete.loc[mask_unmeasured, 'dbh'])
    )

print(df_complete.loc[mask_unmeasured, ['plot', 'row', 'tree', 'dbh', 'ht_final', 'ht_source']]
      .head(10).to_string(index=False))
print(f"... ({mask_unmeasured.sum()} trees estimated in total)")

output_cols = ['plot', 'row', 'tree', 'dbh', 'ht_final', 'ht_source']
df_complete[output_cols].rename(columns={'ht_final': 'ht_m'}).to_csv(
    f"{OUT_DIR}/complete_dataset_with_height.csv", index=False
)
print(f"\nComplete dataset saved: {OUT_DIR}/complete_dataset_with_height.csv")

# -----------------------------------------------------------------------
# TASK 4 — Recompute mean height and dominant height per plot, using
#          the complete dataset, and compare against the (biased)
#          measured-only subsample average
# -----------------------------------------------------------------------
print("\n" + "=" * 70)
print("[Task 4] Mean height and dominant height — complete data vs measured-only")
print("=" * 70)

mean_complete = df_complete.groupby('plot')['ht_final'].mean().rename('mean_ht_complete_m')
mean_measured_only = df_complete[df_complete['ht_measured']].groupby('plot')['ht_final'].mean().rename('mean_ht_measured_only_m')

mean_comparison = pd.concat([mean_complete, mean_measured_only], axis=1)
mean_comparison['difference_m'] = mean_comparison['mean_ht_complete_m'] - mean_comparison['mean_ht_measured_only_m']
mean_comparison['difference_%'] = (mean_comparison['difference_m'] / mean_comparison['mean_ht_measured_only_m']) * 100

print("\n--- Mean height per plot: complete dataset vs measured-only subsample ---")
print(mean_comparison.round(3))
mean_comparison.round(3).to_csv(f"{OUT_DIR}/mean_height_comparison.csv")

# --- Dominant height: compare multiple definitions, then select the
#     most widely used one, as requested by the assignment ---
PLOT_AREA_HA = 0.0441  # from the diameter challenge (Q6 there)
n_dom_assmann = max(1, round(100 * PLOT_AREA_HA))
n_dom_fixed = 6

print(f"\nAssmann's Hdom definition, scaled to plot area ({PLOT_AREA_HA} ha): "
      f"top {n_dom_assmann} largest-DBH trees per plot")


def hdom_top_n(group, n, dbh_col='dbh', ht_col='ht_final'):
    top = group.nlargest(n, dbh_col)
    return top[ht_col].mean()


hdom_assmann = df_complete.groupby('plot').apply(
    lambda g: hdom_top_n(g, n_dom_assmann), include_groups=False
).rename(f'hdom_assmann_top{n_dom_assmann}_m')

hdom_fixed6 = df_complete.groupby('plot').apply(
    lambda g: hdom_top_n(g, n_dom_fixed), include_groups=False
).rename(f'hdom_fixed_top{n_dom_fixed}_m')

hdom_tallest = df_complete.groupby('plot').apply(
    lambda g: hdom_top_n(g, 1), include_groups=False
).rename('hdom_tallest_single_tree_m')

hdom_comparison = pd.concat([hdom_assmann, hdom_fixed6, hdom_tallest], axis=1)
print("\n--- Dominant height: comparison of definitions (using complete dataset) ---")
print(hdom_comparison.round(3))
hdom_comparison.round(3).to_csv(f"{OUT_DIR}/dominant_height_comparison.csv")

print(f"\nSelected definition (most widely used): Assmann's 100 largest-DBH "
      f"trees/ha, scaled to plot size (top {n_dom_assmann} trees/plot here). "
      "This is the internationally standard definition in forest mensuration "
      "and the most common in Brazilian forestry practice/literature.")

hdom_assmann_measured_only = df_complete[df_complete['ht_measured']].groupby('plot').apply(
    lambda g: hdom_top_n(g, min(n_dom_assmann, len(g))), include_groups=False
).rename('hdom_assmann_measured_only_m')

hdom_bias = pd.concat([hdom_assmann, hdom_assmann_measured_only], axis=1)
hdom_bias['difference_m'] = hdom_bias.iloc[:, 0] - hdom_bias.iloc[:, 1]
print("\n--- Assmann Hdom: complete dataset vs measured-only (naive) ---")
print(hdom_bias.round(3))
hdom_bias.round(3).to_csv(f"{OUT_DIR}/dominant_height_bias.csv")

print("\nSCRIPT FINISHED. All outputs written to:", OUT_DIR)